# Financial RAG

## Chunking

Chunking is done by parse_filing.py, then put into the data/ folder.

## Ingestion

In [ ]:
import json
import os
import sys
from pathlib import Path

import psycopg2
import voyageai
from llama_index.core.schema import TextNode
from llama_index.vector_stores.postgres import PGVectorStore

PARSED_DIR = Path("data/parsed")
COMPANIES = ["AAPL", "KO", "LUV"]

EMBED_DIM = 1024
EMBED_MODEL = "voyage-context-4"

PG_HOST = os.environ.get("PG_HOST", "localhost")
PG_PORT = int(os.environ.get("PG_PORT", "5432"))
PG_DATABASE = os.environ.get("PG_DATABASE", "ragdb")
PG_USER = os.environ.get("PG_USER", "raguser")
PG_PASSWORD = os.environ.get("PG_PASSWORD", "ragpass")
PG_TABLE = "finance_filings"

TOKEN_BUDGET = 28_000  # voyage-context-4's real cap is 32K TOTAL tokens, we're just being conservative



In [ ]:
def load_records(company:  str) -> list[dict]:
  path = PARSED_DIR / f"{company}.json"
  return json.loads(path.read_text())

def embed_company(vo: voyageai.Client, records: list[dict]) -> list[list[float]]:
  """One contextualized embed call per company. All company chunks go in as onedocument"""
  chunk_texts = [r["text"] for r in records]
  token_counts = [len(enc.tokens) for enc in vo.tokenize(chunk_texts, model=EMBED_MODEL)]

  batches: list[list[int]] = []  # each entry: indices into records/chunk_texts
  current: list[int] = []
  current_tokens = 0
  for i, t in enumerate(token_counts):
      if current and current_tokens + t > TOKEN_BUDGET:
          batches.append(current)
          current = []
          current_tokens = 0
      current.append(i)
      current_tokens += t
  if current:
      batches.append(current)

  print(f"  {sum(token_counts)} total tokens -> {len(batches)} "
        f"contextualized_embed batch(es) (budget {TOKEN_BUDGET}/batch)")

  embeddings: list[list[float] | None] = [None] * len(records)
  for batch_num, indices in enumerate(batches, 1):
      batch_texts = [chunk_texts[i] for i in indices]
      # NOTE: `output_dimension` is my best guess at the param name for
      # requesting a non-default Matryoshka dimension -- if this errors,
      # drop the kwarg and let it use voyage-context-4's default (1024,
      # which matches EMBED_DIM above anyway, so dropping it costs nothing).
      result = vo.contextualized_embed(
          inputs=[batch_texts],
          model=EMBED_MODEL,
          input_type="document",
          output_dimension=EMBED_DIM,
      )
      batch_embeddings = result.results[0].embeddings
      for idx, emb in zip(indices, batch_embeddings):
          embeddings[idx] = emb
      print(f"    batch {batch_num}/{len(batches)}: {len(indices)} chunks embedded")

  assert all(e is not None for e in embeddings), (
      "some chunk never got an embedding -- batching logic has a gap"
  )
  return embeddings

In [ ]:
def delete_existing(company: str) -> None:
  """Thus function is for idempotency, to wipe the company existing rows before we add a fresh one"""
  conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT, dbname=PG_DATABASE,
    user=PG_USER, password=PG_PASSWORD
  )

  try:
    with conn, conn.cursor() as cur:
      cur.execute(
        f"DELETE FROM data_{PG_TABLE} WHERE metadata_->>'company' = %s",
        (company,),
      )
      print(f" deleted {cur.rowcount} existing ")
  except psycopg2.errors.UndefinedTable:
    print(f"  data_{PG_TABLE} doesn't exist yet (first insert of this "
          f"run will create it) -- nothing to delete")
  finally:
    conn.close()

In [ ]:
companies = ["KO", "LUV"]

In [ ]:
for c in companies:
  if c not in COMPANIES:
    sys.exit(f"Unknown company {c!r}, expected one of {COMPANIES}")

vo = voyageai.Client(api_key=os.environ.get("VOYAGE_API_KEY"))

vector_store = PGVectorStore.from_params(
  host = PG_HOST, port = str(PG_PORT), database = PG_DATABASE,
  user = PG_USER, password = PG_PASSWORD,
  table_name=PG_TABLE, embed_dim = EMBED_DIM,
)

for company in companies:
  print(f"Processing {company}...")
  records = load_records(company)
  print(f"  {len(records)} chunks loaded from {company}.json")

  embeddings = embed_company(vo, records)

  assert len(embeddings) == len(records), (
    f"Got {len(embeddings)} embeddings for {len(records)} chunks"
    "contextualized_embed response shape didn't match what was expected"
  )

  nodes = []

  for i, (record, embedding) in enumerate(zip(records, embeddings)):
    node = TextNode(
      id_=f"{company}_{i}",
      text=record["text"],
      embedding=embedding,
      metadata={
        "company": record["company"],
        "section": record["section"],
        "page": record["page"],
        "fiscal_period": record["fiscal_period"],
        "node_type": record["type"],
      }
    )
    nodes.append(node)

  delete_existing(company=company)
  vector_store.add(nodes)
  print(f"  inserted {len(nodes)} nodes for {company}")

print("Done.")

In [ ]:
from itertools import islice
import json
import numpy as np
from transformers import AutoTokenizer

tokenizer = vo.tokenize

path = Path("data/parsed/KO.json")
with open(path, "r") as file:
  content = file.read()
  items = json.loads(content)
  texts = [item["text"] for item in items]
  tokenize_result = tokenizer(texts) 
  print(tokenize_result)
  num_tokens = np.array([len(i.ids) for i in tokenize_result])
  print(num_tokens.mean(), np.percentile(num_tokens, [25, 50, 75]), f"Max: {num_tokens.max()}")


In [ ]:
def check_datas(company: str) -> None:
  conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT, dbname=PG_DATABASE,
    user=PG_USER, password=PG_PASSWORD
  )

  with conn, conn.cursor() as cur:
    cur.execute(
      f"SELECT * FROM data_{PG_TABLE} WHERE metadata_->>'company' = %s LIMIT 2",
      (company,),
    )
    rows = cur.fetchall()
    for row in rows:
        print(row)

check_datas(company="AAPL")
check_datas(company="KO")
check_datas(company="LUV")


# Query Engine (Retrieval)

In [ ]:
import argparse

from llama_index.core import VectorStoreIndex
from pydantic import PrivateAttr
from llama_index.core.embeddings import BaseEmbedding
from llama_index.core.query_engine import CitationQueryEngine
from llama_index.core.vector_stores import FilterOperator, MetadataFilter, MetadataFilters


COMPANIES = ["AAPL", "KO", "LUV"]
class VoyageContextualQueryEmbedding(BaseEmbedding):
  """Query side-only embed_model wrapper for voyage-context-4"""
  _vo = PrivateAttr()
  _dim = PrivateAttr()

  def __init__(self, model: str = EMBED_MODEL, dim: int = EMBED_DIM, **kwargs):
    super().__init__(model_name = model, **kwargs)
    self._vo = voyageai.Client()
    self._dim = dim

  def _get_query_embedding(self, query: str) -> list[float]:
    result = self._vo.contextualized_embed(
      model = self.model_name,
      inputs=[query],
      input_type="query",
      output_dimension=self._dim,
    )
    return result.results[0].embeddings[0]


  async def _aget_query_embedding(self, query: str) -> list[float]:
    return self._get_query_embedding(query)

  def _get_text_embedding(self, text: str) -> list[float]:
    raise NotImplementedError(
        "VoyageContextualQueryEmbedding is query-only -- document text "
        "should already be pre-embedded in pgvector by build_index.py. "
        "If this fired, something upstream is trying to (re-)embed a "
        "document through the normal LlamaIndex path."
    )
 
  async def _aget_text_embedding(self, text: str) -> list[float]:
      return self._get_text_embedding(text)


def build_index(embed_model: BaseEmbedding):
  vector_store = PGVectorStore.from_params(
      host=PG_HOST, port=str(PG_PORT), database=PG_DATABASE,
      user=PG_USER, password=PG_PASSWORD,
      table_name=PG_TABLE, embed_dim=EMBED_DIM,
  )
  return VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=embed_model)


def company_query_engine(index: VectorStoreIndex, embed_model: BaseEmbedding, company: str) -> CitationQueryEngine:
  filters = MetadataFilters(
    filters =[MetadataFilter(key="company", value=company, operator=FilterOperator.EQ)]
  )
  return CitationQueryEngine.from_args(
    index,
    embed_model=embed_model,
    filter=filters,
    similarity_top_k=4,
    citation_chunk_size=512,
  )

In [ ]:
company = "AAPL"

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI

Settings.llm = GoogleGenAI(model="gemini-3.8-flash")
embed_model = VoyageContextualQueryEmbedding()
Settings.embed_model = embed_model

index = build_index(embed_model)
engine = company_query_engine(index, embed_model=embed_model, company=company)

q1 = "What are apple's main risk factors this period?"
q2 = "What is KO's revenue for the period?"

print(f"q: {q1} \n answer: {engine.query(q1)}\n")
print(f"q: {q2} \n answer: {engine.query(q2)}\n")